In [ ]:
import openmdao.api as om
import numpy as np
from pathlib import Path
import os
import sys
if ".." not in sys.path:
    sys.path.insert(0, "..")


"""import statements"""
from simplerecorder import SimpleRecorder
from mysite import get_setup_params
from aep import AEPCompStochastic, AEPComp
from constraints import SpacingConstraintComp, BoundaryConstraintComp
from aggregator import ConstraintAggregator
from penalty import PenaltyObjectiveComp, FinalRMSViol

from Drivers import NCG, SGD

Loaded: default.csv (1 turbines)
Boundary: 0m x 0m


/Users/brunoboer/Documents/Software/PyWake_new/py_wake/deficit_models/gaussian.py:124: UserWarning: The BastankhahGaussian model is not representative of the setup used in the literature. For this, use py_wake.literature.gaussian_models.Bastankhah_PorteAgel_2014 instead
  DeprecatedModel.__init__(self, 'py_wake.literature.gaussian_models.Bastankhah_PorteAgel_2014')


In [4]:
BASE_DIR = Path.cwd()
RESULTS_DIR = BASE_DIR / "Results"
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

In [3]:
def build_problem(K=50, csv_filename="default.csv", results_dir=RESULTS_DIR, seed=1):
    params = get_setup_params(csv_filename)

    site = params['site']
    turbine = params['turbine']
    wfm = params['wfm']
    x_init = params['x_init']
    y_init = params['y_init']
    boundary_vertices = params['boundary_vertices']
    min_spacing_d = params['min_spacing_d']
    n_turbines = params['n_turbines']
    D = params['diameter']

    min_spacing_m = min_spacing_d * D

    try:
        aep0 = float(wfm(x_init, y_init).aep().sum())
        print(f"Initial AEP: {aep0:.3f} GWh")
    except Exception as e:
        print(f"Initial AEP error: {e}")
        aep0 = None

    prob = om.Problem()
    m = prob.model

    log_path = results_dir / f"run_seed_{seed}.csv"

    recorder = SimpleRecorder(
        prob,
        out_path=log_path,
        x_name='x',
        y_name='y',
        aep_name='aep_comp_deterministic.aep',
        obj_name='objective',
        iter_name='opt_iter',
        viol_name='rms_viol'
    )
    recorder.start()

    indeps = m.add_subsystem('indeps', om.IndepVarComp(), promotes=['*'])
    indeps.add_output('x', val=x_init, units='m')
    indeps.add_output('y', val=y_init, units='m')
    indeps.add_output('opt_iter', val=0.0)

    aep_comp = AEPCompStochastic(
        wake_model=wfm,
        site=site,
        wt_x=x_init,
        wt_y=y_init,
        aep_ref=1.0,
        recorder=None,
        n_cpu=1,
        K=K,
    )

    m.add_subsystem(
        'aep_comp_deterministic',
        AEPComp(
            wake_model=wfm,
            wt_x=x_init,
            wt_y=y_init,
            aep_ref=1.0,
            n_cpu=1,
        ),
        promotes_inputs=['x', 'y']
    )

    m.add_subsystem(
        'aep_comp',
        aep_comp,
        promotes_inputs=['x', 'y'],
        promotes_outputs=['aep'],
    )

    spacing_comp = SpacingConstraintComp(
        n_turbines=n_turbines,
        min_spacing=min_spacing_m,
        eps=1e-12,
    )
    m.add_subsystem(
        'spacing_comp',
        spacing_comp,
        promotes_inputs=['x', 'y'],
        promotes_outputs=['spacing_cons'],
    )

    boundary_comp = BoundaryConstraintComp(
        boundary_vertices=boundary_vertices,
        n_turbines=n_turbines,
    )
    m.add_subsystem(
        'boundary_comp',
        boundary_comp,
        promotes_inputs=['x', 'y'],
        promotes_outputs=['boundary_cons'],
    )

    agg_comp = ConstraintAggregator(n_turbines=n_turbines)
    m.add_subsystem(
        'constraint_agg',
        agg_comp,
        promotes_inputs=['spacing_cons', 'boundary_cons'],
        promotes_outputs=['g_vector'],
    )

    penalty_comp = PenaltyObjectiveComp(n_constraints=agg_comp.m_total)
    m.add_subsystem(
        'penalty_comp',
        penalty_comp,
        promotes_inputs=['aep', 'g_vector'],
        promotes_outputs=['objective', 'penalty'],
    )

    nc = agg_comp.m_total
    m.add_subsystem(
        'rms_viol',
        FinalRMSViol(nconstraints=nc),
        promotes_outputs=['rms_viol']
    )
    m.connect('g_vector', 'rms_viol.g_vector')

    prob.driver = NCG(maxiter=30)
    prob.driver.options['learning_rate'] = params['diameter'] / 5.0
    prob.driver.options['gamma_min'] = 0.2 * (params['diameter'] / 5.0)
    prob.driver.options['lower'] = 1e-6
    prob.driver.options['upper'] = 1e-1
    prob.driver.options['tol'] = 1e-6
    prob.driver.options['disp'] = True

    m.add_design_var(
        'x',
        lower=boundary_vertices[:, 0].min(),
        upper=boundary_vertices[:, 0].max()
    )
    m.add_design_var(
        'y',
        lower=boundary_vertices[:, 1].min(),
        upper=boundary_vertices[:, 1].max()
    )

    m.add_objective('aep', scaler=-1.0)
    m.add_constraint('penalty', lower=0.0, scaler=1.0)

    return prob, recorder, log_path

In [4]:
def main(csv_filename="default.csv", K=50, seed=10):
    csv_filename = f"100turb_3600m_kdt_{seed}.csv"

    prob, recorder, log_path = build_problem(
        K=K,
        csv_filename=csv_filename,
        results_dir=RESULTS_DIR,
        seed=seed
    )

    prob.setup()
    prob.run_driver()

    return prob, recorder, log_path

In [ ]:
prob, recorder, log_path = main(seed=1, K=50)

Loaded: 100turb_3600m_kdt_1.csv (100 turbines)
Boundary: 0m x 3600m
Initial AEP: 496.257 GWh


In [ ]:
print("Log save in:", log_path)
print("AEP final:", prob.get_val('aep'))
print("Penalty final:", prob.get_val('penalty'))
print("RMS viol final:", prob.get_val('rms_viol'))
print("x final:", prob.get_val('x'))
print("y final:", prob.get_val('y'))